# Analysis of JV Curves: Experimental Data vs SCAPS Simulations

## Overview
This notebook automates the analysis and comparison of J-V (current density-voltage) curves from laboratory measurements with those simulated using SCAPS (Solar Cell Capacitance Simulator). The goal is to validate the photovoltaic device model by comparing experimental performance with multiple simulation scenarios, identifying which simulations best match the measured behavior through parameter-based and shape-based matching algorithms.

## Section 1: Configuration and Target Parameters

In this section, we define the target performance metrics and experimental calibration parameters. These values represent the expected or desired photovoltaic device performance and are used to:
- **Normalize** the device's electrical characteristics
- **Convert** experimental current to current density
- **Set reference points** for comparing simulations against experiments

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.interpolate import interp1d
from sklearn.metrics import mean_squared_error
import warnings
import os
warnings.filterwarnings('ignore')

# Create plots directory if it doesn't exist
plots_dir = 'Plots'
if not os.path.exists(plots_dir):
    os.makedirs(plots_dir)
    print(f"✅ Created '{plots_dir}' directory for saving plots")

# -------------------------------------------------------------
# 1. DEFINE YOUR TARGET VALUES HERE
# -------------------------------------------------------------
TARGET_VOC = 0.38   # Volts (V)
TARGET_JSC = 20.31    # mA/cm2
TARGET_FF  = 49.56    # %
TARGET_ETA = 4.57     # %

# -------------------------------------------------------------
# 2. EXPERIMENTAL CALIBRATION
# -------------------------------------------------------------
# Define your cell's active area in cm² to convert I to J (mA/cm²)
ACTIVE_AREA_CM2 = 0.1 #this will be calculated after, is just a placeholder for now

# SCAPS typically outputs negative photocurrent.
# If your experimental measurement yielded positive current values, 
# the code automatically inverts them to make both curves comparable.
INVERT_EXP_CURRENT = True

## Theoretical Foundation: The Single-Diode Model

### Overview
The behavior of photovoltaic devices under illumination is commonly modeled using the **single-diode equivalent circuit**, which provides a physical interpretation of J-V curves. This model connects microscopic device physics to macroscopic electrical measurements.

### Mathematical Framework

The current-voltage relationship in solar cells is described by the **Shockley ideal diode equation**:

$$J = J_L - J_0 \left[\exp\left(\frac{qV}{nk_BT}\right) - 1\right] - \frac{V}{R_{sh}}$$

where:
- $J$ = current density (mA/cm²)
- $J_L$ = light-generated current density (photocurrent)
- $J_0$ = dark saturation current density (reverse saturation)
- $q$ = elementary charge (1.602 × 10⁻¹⁹ C)
- $V$ = applied voltage (V)
- $n$ = ideality factor (typically 1-2, accounts for recombination mechanisms)
- $k_B$ = Boltzmann constant (1.381 × 10⁻²³ J/K)
- $T$ = absolute temperature (K, typically 298 K at STC)
- $R_{sh}$ = shunt resistance (parallel resistance, Ω·cm²)

### Physical Interpretation

1. **First Term $(J_L)$**: Represents photocurrent generated by absorbed photons. Independent of voltage.

2. **Second Term $(J_0[\exp(...)-1])$**: Describes dark current from thermal generation and recombination. Exponentially increases with voltage above thermal voltage $V_T = \frac{k_BT}{q} \approx 26$ mV at 298 K.

3. **Third Term $(V/R_{sh})$**: Leakage current due to shunt paths; becomes significant at high voltages or low light conditions.

### Under Illumination: Modified Equation

Under illumination, the net current is:

$$J = J_L - J_0 \left[\exp\left(\frac{q(V + J \cdot R_s)}{nk_BT}\right)\right] - \frac{V + J \cdot R_s}{R_{sh}}$$

This includes **series resistance** $R_s$ (resistance of the contacts, semiconductor, and grid), which causes an additional voltage drop proportional to current.

### Key Performance Metrics

At the light-dark current crossover:

$$\boxed{\text{At } V_{oc}: J = 0 \implies J_L = J_0 \left[\exp\left(\frac{qV_{oc}}{nk_BT}\right) - 1\right]}$$

The **Fill Factor** represents the ratio of the maximum power rectangle to the theoretical maximum:

$$FF = \frac{P_{max}}{V_{oc} \cdot J_{sc}} = \frac{V_{mp} \cdot J_{mp}}{V_{oc} \cdot J_{sc}}$$

where $(V_{mp}, J_{mp})$ is the operating point of maximum power output.

**Efficiency** (Power Conversion Efficiency):

$$\eta = \frac{P_{max}}{P_{in}} = \frac{V_{oc} \cdot J_{sc} \cdot FF}{P_{in}}$$

where $P_{in}$ is incident solar power (typically 100 mW/cm² under STC).

### How Series and Shunt Resistance Affect Curve Shape

- **High $R_s$**: Reduces FF, causes "S-shape" at high voltages; limits maximum power
- **Low $R_{sh}$**: Reduces $V_{oc}$, increases leakage at forward bias; evident as steep slope near $J_{sc}$
- **Ideal case**: $R_s \to 0$, $R_{sh} \to \infty$, resulting in sharp rectangular J-V curve

### SCAPS and the Single-Diode Model

SCAPS solves the complete semiconductor transport equations without assuming the single-diode form. However, by analyzing the resulting J-V curves, SCAPS-simulated data can be fitted to extract effective values of $J_L$, $J_0$, $n$, $R_s$, and $R_{sh}$, providing a connection between microscopic device parameters (layer thickness, doping, defects) and macroscopic circuit behavior.

## Section 2: Experimental Data Reading and Visualization

### Background: The J-V Curve
The current density-voltage (J-V) curve is a fundamental characterization of photovoltaic devices. It shows how the current output of a solar cell varies with applied voltage under steady illumination. Key parameters extracted from the J-V curve include:

- **$V_{oc}$ (Open Circuit Voltage)**: Maximum voltage output when no current flows  
- **$J_{sc}$ (Short Circuit Current Density)**: Maximum current when voltage is zero  
- **$FF$ (Fill Factor)**: Ratio of maximum power to the product of $V_{oc}$ and $J_{sc}$  
- **$\eta$ (Power Conversion Efficiency)**: Percentage of incident solar power converted to electrical power

In this section, we read experimental I-V data from the solar simulator measurements and prepare it for comparison with simulations.

In [2]:
# -------------------------------------------------------------
# PLOT 1: READ EXPERIMENTAL DATA & CONVERT TO JV CURVE
# -------------------------------------------------------------

# File paths (Adjust these to match your actual file names in the folder)
exp_file = "C:/Users/jp_ol/OneDrive/Ambiente de Trabalho/TESE/Data_Analysis/data/CdS3_1/CdS3_1_IV Graph.xlsx"
results_file = "C:/Users/jp_ol/OneDrive/Ambiente de Trabalho/TESE/Data_Analysis/data/CdS3_1/CdS3_1_Results Table.xlsx"

# The specific measurement you want to plot
col_name = "CdS3_6 2026-01-22 14-24-51"

# 1. READ RESULTS TABLE TO CALCULATE EXACT ACTIVE AREA
try:
    results_df = pd.read_excel(results_file, engine='openpyxl')
    
    print(f"📊 Available measurements in Results Table:")
    for i, meas in enumerate(results_df['Measurement'].values):
        print(f"   {i}: {meas}")
    
    # Locate the specific measurement in the Results Table
    target_row = results_df[results_df['Measurement'] == col_name]
    
    if not target_row.empty:
        isc_a = target_row['Isc A'].values[0]
        jsc_ma = target_row['Jsc mA/cm2'].values[0]
        
        # Calculate Active Area: (Isc / Jsc) * 1000
        calculated_area = (isc_a / jsc_ma) * 1000
        print(f"\n✅ Target measurement '{col_name}' found in Results Table.")
        print(f"📐 Dynamically calculated Active Area: {calculated_area:.4f} cm²")
    else:
        print(f"\n⚠️ Measurement '{col_name}' NOT FOUND in Results Table!")
        print(f"   Please update 'col_name' variable to match one of the measurements above.")
        print(f"   Using fallback area: {ACTIVE_AREA_CM2} cm²")
        calculated_area = ACTIVE_AREA_CM2
except Exception as e:
    print(f"❌ Error reading Results Table: {type(e).__name__}: {e}")
    print(f"   Using fallback area: {ACTIVE_AREA_CM2} cm²")
    calculated_area = ACTIVE_AREA_CM2

# 2. READ IV GRAPH & CONVERT TO JV CURVE
exp_df_raw = pd.read_excel(exp_file, header=None)
col_idx = None
for i in range(len(exp_df_raw.columns)):
    if col_name in str(exp_df_raw.iloc[0, i]):
        col_idx = i
        break

if col_idx is None:
    raise ValueError(f"Column '{col_name}' not found in {exp_file}")

# Skip the 1st row (which contains the string headers 'Vmeas' and 'Imeas')
# Row 0: measurement label, Row 1: column type labels, Row 2+: data
v_exp_raw = pd.to_numeric(exp_df_raw.iloc[2:, col_idx], errors='coerce')
i_exp_raw = pd.to_numeric(exp_df_raw.iloc[2:, col_idx + 1], errors='coerce')

# Remove NaN values
valid_mask = pd.notna(v_exp_raw) & pd.notna(i_exp_raw)
v_exp = v_exp_raw[valid_mask].values
i_exp = i_exp_raw[valid_mask].values

# Convert Current (A) to Current Density (mA/cm2) using the dynamically calculated area
j_exp = (i_exp * 1000) / calculated_area

# Invert current if necessary (to match SCAPS typical output or 1st quadrant representation)
if INVERT_EXP_CURRENT:
    j_exp = -j_exp

# Create a clean DataFrame
exp_data = pd.DataFrame({'v(V)': v_exp, 'jtot(mA/cm2)': j_exp}).dropna()

# Apply correction factor to experimental current density from EQE
exp_data['jtot(mA/cm2)'] = exp_data['jtot(mA/cm2)'] * 1.150988311

# 3. PLOT EXPERIMENTAL JV CURVE
fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=exp_data['v(V)'], 
    y=exp_data['jtot(mA/cm2)'], 
    mode='lines', name='Experimental (Converted to J)', 
    line=dict(color='black', dash='dash', width=3)
))
fig1.update_xaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
fig1.update_yaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
fig1.update_layout(
    title=f"Experimental JV Curve ({col_name})",
    xaxis_title='Voltage (V)',
    yaxis_title='Current Density (mA/cm²)',
    template='plotly_white', width=900, height=600,
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridwidth=1, gridcolor='lightgray'),
    yaxis=dict(showgrid=True, gridwidth=1, gridcolor='lightgray')
)
fig1.show()
fig1.write_image(os.path.join(plots_dir, 'Experimental_JV.png'), width=900, height=600)
print(f"✅ Saved Plot 1 to {plots_dir}/Experimental_JV.png")


📊 Available measurements in Results Table:
   0: CdS3_1 2026-01-22 14-08-37
   1: CdS3_2 2026-01-22 14-13-57
   2: CdS3_3 2026-01-22 14-18-00
   3: CdS3_4 2026-01-22 14-19-58
   4: CdS3_5 2026-01-22 14-22-04
   5: CdS3_6 2026-01-22 14-24-51
   6: CdS3_7 2026-01-22 14-26-23
   7: CdS3_8 2026-01-22 14-28-56
   8: CdS3_9 2026-01-22 14-35-21
   9: CdS3_10 2026-01-22 14-36-43
   10: CdS3_11 2026-01-22 14-38-31
   11: CdS3_12 2026-01-22 14-40-38
   12: CdS3_13 2026-01-22 14-45-48
   13: CdS3_14 2026-01-22 14-47-32
   14: CdS3_15 2026-01-22 14-49-23
   15: CdS3_16 2026-01-22 14-50-52
   16: CdS3_17 2026-01-22 14-52-17
   17: CdS3_18 2026-01-22 14-53-58
   18: CdS3_19 2026-01-22 14-55-56
   19: CdS3_20 2026-01-22 14-57-24
   20: CdS3_21 2026-01-22 14-58-57
   21: CdS3_22 2026-01-22 15-00-42
   22: CdS3_23 2026-01-22 15-02-16
   23: CdS3_24 2026-01-22 15-03-33
   24: CdS3_25 2026-01-22 15-04-41

✅ Target measurement 'CdS3_6 2026-01-22 14-24-51' found in Results Table.
📐 Dynamically calculated A

✅ Saved Plot 1 to Plots/Experimental_JV.png


## Section 3: SCAPS Simulations Overview

### What is SCAPS?
**SCAPS** (Solar Cell Capacitance Simulator) is a specialized software tool developed at Ghent University for simulating the electrical behavior of thin-film photovoltaic devices. It solves Poisson's equation and the continuity equations in a multi-layer semiconductor structure to predict device performance under certain test conditions.

### SCAPS Simulation Methodology
SCAPS performs detailed numerical simulations of solar cells by:

1. **Device Architecture Definition**: Specifies material layers, thickness, doping levels, and defect parameters
2. **Physical Parameters**: Sets electron and hole transport properties, recombination mechanisms, and optical characteristics
3. **Numerical Solution**: Solves coupled differential equations:
   - **Poisson Equation**: $\frac{d^2\phi}{dx^2} = -\frac{q}{\varepsilon_0\varepsilon_r}(p - n + N_D^+ - N_A^-)$
   - **Continuity Equations**: 
     - Electrons: $\frac{\partial n}{\partial t} = \frac{1}{q}\frac{\partial J_n}{\partial x} - (U - G)$
     - Holes: $\frac{\partial p}{\partial t} = - \frac{1}{q}\frac{\partial J_p}{\partial x} - (U - G)$

   where $\phi$ is electrostatic potential, $q$ is elementary charge, $n$ and $p$ are electron and hole concentrations, $J_n$ and $J_p$ are current densities, $G$ is generation rate, and $U$ is recombination rate.

4. **I-V Characteristics**: Calculates the J-V curve by sweeping applied voltage and computing resulting current

### Key Physical Processes in SCAPS
- **Photocurrent Generation**: Incident photons with energy > bandgap create electron-hole pairs
- **Carrier Transport**: Electrons and holes drift under electric field and diffuse down concentration gradients
- **Recombination**: Losses from multiple recombination mechanisms (radiative, Auger, defect-assisted SRH)
- **Interface Effects**: Band alignment, interfacial defects, and charge extraction barriers

### Output from SCAPS
SCAPS generates J-V curves showing current density (mA/cm²) as a function of applied voltage (V), along with derived parameters: $V_{oc}$, $J_{sc}$, and efficiency $\eta$. Multiple simulations with varied parameters (e.g., thickness, doping, defect density) help identify optimal device configurations.

In this Simulation standard conditions were used (STC: 1000 W/m², AM1.5 spectrum, 25°C).

In [3]:
# Helper function to read SCAPS .iv file
def read_scaps_iv_file(file_path):
    """
    Parse SCAPS .iv file with tab-separated or fixed-width columns.
    Returns a list of simulations with their data.
    """
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    
    # Split by major sections: each simulation starts with "Batch parameters"
    sections = content.split('**Batch parameters**')
    simulations = []
    
    for section_idx, section in enumerate(sections[1:]):  # Skip first empty section
        sim_dict = {}
        lines = section.split('\n')
        
        # Extract ALL batch parameters (first few non-empty lines after header)
        batch_params = []
        for i in range(min(5, len(lines))):  # Check first 5 lines for parameters
            param_line = lines[i].strip()
            if not param_line:
                continue
            # Stop if we hit the data table header
            if 'v(V)' in param_line:
                break
            # Extract parameter
            if ':' in param_line:
                parts = param_line.split(':', 1)  # Split only on first colon
                if len(parts) >= 2:
                    param_name = parts[0].strip()
                    param_val = parts[1].strip()
                    batch_params.append((param_name, param_val))
        
        sim_dict['batch_params'] = batch_params
        
        # Find the data table header (line with "v(V)" and "jtot")
        data_start = -1
        for i, line in enumerate(lines):
            if 'v(V)' in line and 'jtot' in line:
                data_start = i + 1
                break
        
        if data_start == -1:
            continue
        
        # Extract data lines AND performance parameters
        iv_data = {'v': [], 'j': []}
        voc = jsc = ff = eta = np.nan
        
        for i in range(data_start, len(lines)):
            line = lines[i].strip()
            
            # Skip empty lines
            if not line:
                continue
            
            # Check for performance parameters
            if 'Voc' in line and '=' in line:
                try:
                    # Extract value after "="
                    value_part = line.split('=')[1].strip()
                    voc = float(value_part.split()[0])
                except (ValueError, IndexError):
                    pass
                continue
            
            if 'Jsc' in line and '=' in line:
                try:
                    value_part = line.split('=')[1].strip()
                    jsc = float(value_part.split()[0])
                except (ValueError, IndexError):
                    pass
                continue
            
            if 'FF' in line and '=' in line:
                try:
                    value_part = line.split('=')[1].strip()
                    ff = float(value_part.split()[0])
                except (ValueError, IndexError):
                    pass
                continue
            
            if 'eta' in line and '=' in line:
                try:
                    value_part = line.split('=')[1].strip()
                    eta = float(value_part.split()[0])
                except (ValueError, IndexError):
                    pass
                continue
            
            # Try to parse as numerical data (voltage/current pair)
            try:
                # Split by whitespace (handles both spaces and tabs)
                parts = line.split()
                if len(parts) >= 2:
                    # Try to parse first two columns as v and j
                    v_val = float(parts[0])
                    j_val = float(parts[1])
                    iv_data['v'].append(v_val)
                    iv_data['j'].append(j_val)
            except (ValueError, IndexError):
                # Not a data line, skip
                pass
        
        # Store data
        sim_dict['v(V)'] = np.array(iv_data['v'])
        sim_dict['jtot(mA/cm2)'] = np.array(iv_data['j'])
        sim_dict['Voc'] = voc
        sim_dict['Jsc'] = jsc
        sim_dict['FF'] = ff
        sim_dict['eta'] = eta
        
        # Create readable label - extract just the value from batch parameter
        # Format: "Defect Density = 1e+10" or similar
        if batch_params:
            param_name = batch_params[0][0]
            param_val = batch_params[0][1]
            
            # Extract just the numerical value and unit if present
            # e.g., "1.000e+10" from the value
            # Try to get just the number part
            val_parts = param_val.split()
            if val_parts:
                numeric_val = val_parts[0]  # Get the first part (the number)
            else:
                numeric_val = param_val
            
            # Extract the key parameter name (e.g., "total defect density")
            if '>>' in param_name:
                # Get the last part after >>
                key_name = param_name.split('>>')[-1].strip()
            else:
                key_name = param_name
            
            # Shorten the parameter name
            if 'total defect density' in key_name.lower():
                key_name = 'Defect Density'
            
            label = f"{key_name} = {numeric_val}"
        else:
            label = f"Simulation {section_idx+1}"
        
        sim_dict['label'] = label
        
        # Only add if we have data
        if len(iv_data['v']) > 0:
            simulations.append(sim_dict)
    
    return simulations

# Read the SCAPS file
sim_file = "C:/Users/jp_ol/OneDrive/Ambiente de Trabalho/TESE/Data_Analysis/data/CdS3_1/CdS3_1_CdS_N_e_INTER.iv"
scaps_data = read_scaps_iv_file(sim_file)

# Format for downstream use
if len(scaps_data) == 0:
    print("⚠️ No SCAPS simulation data found in the file.")
else:
    scaps_data_formatted = []
    for sim in scaps_data:
        scaps_data_formatted.append({
            'label': sim['label'],
            'batch_params': sim['batch_params'],
            'v(V)': sim['v(V)'],
            'jtot(mA/cm2)': sim['jtot(mA/cm2)'],
            'Voc': sim['Voc'],
            'Jsc': sim['Jsc'],
            'FF': sim['FF'],
            'eta': sim['eta']
        })
    scaps_data = scaps_data_formatted
    
    # RESULTS TABLE
    print("\n" + "="*110)
    print("=== EXTRACTED SCAPS PARAMETERS SUMMARY ===")
    print("="*110)
    summary_df = pd.DataFrame([{
        'Simulation ID': d['label'], 
        'Voc (V)': f"{d['Voc']:.6f}", 
        'Jsc (mA/cm²)': f"{d['Jsc']:.4f}", 
        'FF (%)': f"{d['FF']:.4f}", 
        'η (%)': f"{d['eta']:.4f}"
    } for d in scaps_data])
    display(summary_df)
    print("="*110)

    # Compute experimental limits
    EXP_V_MAX = exp_data['v(V)'].max()
    EXP_J_MAX = exp_data['jtot(mA/cm2)'].max()

    # PLOT 2: ALL SIMULATIONS
    fig2 = go.Figure()
    for i, d in enumerate(scaps_data):
        v_filtered = d['v(V)']
        j_filtered = d['jtot(mA/cm2)']
        
        hover_text = f"<b>{d['label']}</b><br>Voc: {d['Voc']:.6f} V<br>Jsc: {d['Jsc']:.4f} mA/cm²<br>FF: {d['FF']:.4f} %<br>η: {d['eta']:.4f} %"
        legend_name = f"Sim {i+1}: {d['label']}"
        
        fig2.add_trace(go.Scatter(
            x=v_filtered, y=j_filtered, mode='lines', 
            name=legend_name, hoverinfo='text', hovertext=hover_text
        ))

    fig2.update_xaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
    fig2.update_yaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
    fig2.update_layout(
        title='All SCAPS Simulation IV Curves',
        xaxis_title='Voltage (V)', yaxis_title='Current Density (mA/cm2)',
        hovermode='closest', template='plotly_white', width=1000, height=600,
        plot_bgcolor='white',
        xaxis=dict(showgrid=True, gridwidth=1, gridcolor='lightgray'),
        yaxis=dict(showgrid=True, gridwidth=1, gridcolor='lightgray')
    )
    fig2.show()
    fig2.write_image(os.path.join(plots_dir, 'All_Simulations.png'), width=1000, height=600)
    print(f"\n✅ Saved Plot 2 to {plots_dir}/All_Simulations.png")



=== EXTRACTED SCAPS PARAMETERS SUMMARY ===


,Simulation ID,Voc (V),Jsc (mA/cm²),FF (%),η (%)
0,Defect Density = 1.000e+12,0.423869,23.6365,52.9705,5.3070
1,Defect Density = 1.000e+12,0.423484,23.6365,52.8904,5.2942
2,Defect Density = 1.000e+12,0.421945,23.6363,52.5948,5.2454
3,Defect Density = 1.000e+12,0.420019,23.6361,52.2732,5.1895
4,Defect Density = 1.000e+12,0.404803,23.6346,50.8507,4.8651
5,Defect Density = 1.000e+12,0.389283,23.6326,50.1632,4.6149
6,Defect Density = 1.000e+12,0.330804,23.6170,48.7410,3.8079
7,Defect Density = 1.000e+13,0.423869,23.6365,52.9704,5.3070
8,Defect Density = 1.000e+13,0.423484,23.6365,52.8904,5.2942
9,Defect Density = 1.000e+13,0.421945,23.6363,52.5948,5.2454



✅ Saved Plot 2 to Plots/All_Simulations.png


## Section 3.1: Overlay Comparison - All Simulations vs Experimental Data

### Overview
This section provides a comprehensive visual comparison by overlaying all SCAPS simulation J-V curves on the same plot as the experimental data. This overlay approach allows for:

- **Direct shape comparison**: Visually assess which simulations best match the experimental curve shape
- **Parameter range assessment**: Understand the spread of voltages, currents, and fill factors across all simulations
- **Quality of model fit**: Quickly identify if any simulations closely track the experimental behavior across the full J-V range
- **Anomaly detection**: Spot simulations that deviate significantly from the general trend

This visualization complements the parameter-based matching in the following section and provides intuitive insight into device behavior.

In [4]:
# ------------------ PLOT 3.1: ALL SIMULATIONS + EXPERIMENTAL OVERLAY ------------------
fig_overlay = go.Figure()

# Add experimental data first (so it appears at the back)
fig_overlay.add_trace(go.Scatter(
    x=exp_data['v(V)'], 
    y=exp_data['jtot(mA/cm2)'], 
    mode='lines', 
    name='Experimental Data', 
    line=dict(color='black', width=3, dash='dash'),
    hovertemplate='<b>Experimental</b><br>V: %{x:.3f} V<br>J: %{y:.2f} mA/cm²<extra></extra>'
))

# Add all SCAPS simulations
colors = []
import plotly.express as px
color_palette = px.colors.qualitative.Plotly
for i, d in enumerate(scaps_data):
    v_filtered = d['v(V)']
    j_filtered = d['jtot(mA/cm2)']
    
    # Hover information
    hover_text = f"<b>{d['label']}</b><br>Voc: {d['Voc']:.3f} V<br>Jsc: {d['Jsc']:.2f} mA/cm²<br>FF: {d['FF']:.1f} %<br>η: {d['eta']:.2f} %"
    
    # Alternate between solid and dash line styles for better visibility
    line_dash = 'solid' if i % 2 == 0 else 'dash'
    line_color = color_palette[i % len(color_palette)]
    
    fig_overlay.add_trace(go.Scatter(
        x=v_filtered, 
        y=j_filtered, 
        mode='lines', 
        name=f"Sim {i+1}",
        line=dict(color=line_color, width=2, dash=line_dash),
        hovertemplate=hover_text + '<extra></extra>'
    ))

fig_overlay.update_xaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
fig_overlay.update_yaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
fig_overlay.update_layout(
    title='All SCAPS Simulations Overlaid with Experimental JV Curve',
    xaxis_title='Voltage (V)',
    yaxis_title='Current Density (mA/cm²)',
    hovermode='closest',
    template='plotly_white',
    width=1100,
    height=650,
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridwidth=1, gridcolor='lightgray'),
    yaxis=dict(showgrid=True, gridwidth=1, gridcolor='lightgray'),
    legend=dict(
        x=1.02,
        y=1,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.7)',
        bordercolor='gray',
        borderwidth=1
    )
)
fig_overlay.show()
fig_overlay.write_image(os.path.join(plots_dir, 'All_data_Comparison.png'), width=1100, height=650)

print(f"\n✅ Overlay plot created with {len(scaps_data)} simulations + 1 experimental curve")
print(f"✅ Saved Plot 3.1 to {plots_dir}/All_data_Comparison.png")
print(f"   Experimental range: V ∈ [{exp_data['v(V)'].min():.3f}, {exp_data['v(V)'].max():.3f}] V")
print(f"                       J ∈ [{exp_data['jtot(mA/cm2)'].min():.2f}, {exp_data['jtot(mA/cm2)'].max():.2f}] mA/cm²")



✅ Overlay plot created with 56 simulations + 1 experimental curve
✅ Saved Plot 3.1 to Plots/All_data_Comparison.png
   Experimental range: V ∈ [-0.100, 0.500] V
                       J ∈ [-20.86, 53.47] mA/cm²


## Section 4: Parameter-Based Comparison with Target Values

### Methodology
In this section, we compare simulations against **target performance parameters** using a scoring algorithm. For each simulation, we calculate a normalized distance metric:

$$\text{Score} = \sum_{i} \left(\frac{X_i^{sim} - X_i^{target}}{X_i^{target}}\right)^2$$

where $X_i$ represents the performance parameters:
- $V_{oc}$ (Open Circuit Voltage)
- $J_{sc}$ (Short Circuit Current Density)  
- $FF$ (Fill Factor)
- $\eta$ (Power Conversion Efficiency)

**Interpretation**: Lower scores indicate simulations whose parameters are closer to the target values. This approach is useful for identifying device designs that achieve desired specifications.

### Application
The top 5 simulations with the lowest parameter scores are plotted alongside experimental data to evaluate whether parameter matching correlates with realistic device behavior.

In [5]:
# Calculate a Score (the lower the score, the closer to the target parameters)
for d in scaps_data:
    score = 0
    score += ((d['Voc'] - TARGET_VOC) / TARGET_VOC)**2
    score += ((d['Jsc'] - TARGET_JSC) / TARGET_JSC)**2
    score += ((d['FF'] - TARGET_FF) / TARGET_FF)**2
    score += ((d['eta'] - TARGET_ETA) / TARGET_ETA)**2
    d['param_score'] = score

top_5_params = sorted(scaps_data, key=lambda x: x['param_score'])[:5]

# ------------------ PLOT 3: TOP 5 VS EXPERIMENTAL (PARAMETERS) ------------------
fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=exp_data['v(V)'], y=exp_data['jtot(mA/cm2)'], 
    mode='lines', name='Experimental', line=dict(color='black', dash='dash', width=3)
))

for i, d in enumerate(top_5_params):
    v_filtered = d['v(V)']
    j_filtered = d['jtot(mA/cm2)']
    
    hover_text = f"<b>{d['label']}</b><br>Voc: {d['Voc']} V<br>Jsc: {d['Jsc']} mA/cm²<br>FF: {d['FF']} %<br>η: {d['eta']} %"
    fig3.add_trace(go.Scatter(
        x=v_filtered, y=j_filtered, mode='lines', 
        name=f"Top {i+1} (Parameter Score)", hoverinfo='text', hovertext=hover_text
    ))

fig3.update_xaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
fig3.update_yaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
fig3.update_layout(
    title='Top 5 Simulations Closest to Target Parameters',
    xaxis_title='Voltage (V)', yaxis_title='Current Density (mA/cm2)',
    hovermode='closest', template='plotly_white', width=900, height=600,
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridwidth=1, gridcolor='lightgray'),
    yaxis=dict(showgrid=True, gridwidth=1, gridcolor='lightgray')
)
fig3.show()
fig3.write_image(os.path.join(plots_dir, 'Top5_Parameters.png'), width=900, height=600)
print(f"✅ Saved Plot 3 to {plots_dir}/Top5_Parameters.png")

# Create comparison table
print("\n" + "="*120)
print("TOP 5 PARAMETER MATCHES: EXPERIMENTAL VS SIMULATIONS")
print("="*120)
print(f"\n{'Metric':<20} {'Experimental':<18} {'Top 1':<18} {'Top 2':<18} {'Top 3':<18} {'Top 4':<18} {'Top 5':<18}")
print("-"*120)
print(f"{'Voc (V)':<20} {TARGET_VOC:<18.4f} {top_5_params[0]['Voc']:<18.4f} {top_5_params[1]['Voc']:<18.4f} {top_5_params[2]['Voc']:<18.4f} {top_5_params[3]['Voc']:<18.4f} {top_5_params[4]['Voc']:<18.4f}")
print(f"{'Jsc (mA/cm²)':<20} {TARGET_JSC:<18.4f} {top_5_params[0]['Jsc']:<18.4f} {top_5_params[1]['Jsc']:<18.4f} {top_5_params[2]['Jsc']:<18.4f} {top_5_params[3]['Jsc']:<18.4f} {top_5_params[4]['Jsc']:<18.4f}")
print(f"{'FF (%)':<20} {TARGET_FF:<18.2f} {top_5_params[0]['FF']:<18.2f} {top_5_params[1]['FF']:<18.2f} {top_5_params[2]['FF']:<18.2f} {top_5_params[3]['FF']:<18.2f} {top_5_params[4]['FF']:<18.2f}")
print(f"{'η (%)':<20} {TARGET_ETA:<18.4f} {top_5_params[0]['eta']:<18.4f} {top_5_params[1]['eta']:<18.4f} {top_5_params[2]['eta']:<18.4f} {top_5_params[3]['eta']:<18.4f} {top_5_params[4]['eta']:<18.4f}")
print(f"{'Param Score':<20} {'---':<18} {top_5_params[0]['param_score']:<18.6f} {top_5_params[1]['param_score']:<18.6f} {top_5_params[2]['param_score']:<18.6f} {top_5_params[3]['param_score']:<18.6f} {top_5_params[4]['param_score']:<18.6f}")
print("="*120)


✅ Saved Plot 3 to Plots/Top5_Parameters.png

TOP 5 PARAMETER MATCHES: EXPERIMENTAL VS SIMULATIONS

Metric               Experimental       Top 1              Top 2              Top 3              Top 4              Top 5             
------------------------------------------------------------------------------------------------------------------------
Voc (V)              0.3800             0.3994             0.4156             0.4179             0.4198             0.4202            
Jsc (mA/cm²)         20.3100            21.9596            21.9611            21.9613            21.9614            21.9615           
FF (%)               49.56              48.36              49.27              49.48              49.69              49.75             
η (%)                4.5700             4.2417             4.4967             4.5405             4.5808             4.5910            
Param Score          ---                0.014940           0.015655           0.016582           0.017580

## Section 5: Shape-Based Comparison Using Mean Squared Error (MSE)

### Motivation
While parameter-based matching identifies designs meeting target specifications, it does not guarantee realistic device physics. Two J-V curves can have identical parameters but very different shapes, indicating different dominant loss mechanisms. This section employs a **shape-matching algorithm** to identify simulations whose curve morphology most closely resembles the experimental measurement.

### Mathematical Approach
For each simulation, we:

1. **Interpolate** the simulated J-V curve to the exact voltages measured in the experiment
2. **Calculate Mean Squared Error** in the overlapping voltage range:

$$\text{MSE} = \frac{1}{N} \sum_{i=1}^{N} \left(J_i^{exp} - J_i^{sim}(V_i)\right)^2$$

where $J^{exp}$ is experimental current density at measured voltages, $J^{sim}(V_i)$ is interpolated simulated current density at those same voltages, and $N$ is the number of measurement points.

3. **Identify** the top 5 simulations with **lowest MSE** values

### Interpretation
Lower MSE values indicate better agreement in curve shape. This metric is sensitive to:
- The "knee" or transition region of the J-V curve (determined by series/shunt resistance and ideality factors)
- Overall current deficit and voltage losses
- Non-ideal behavior like S-shaped curves or kinks caused by interface barriers

### Comparison with Parameter-Based Matching
- **Parameter-based**: Focuses on specific values ($V_{oc}$, $J_{sc}$, $FF$, $\eta$)
- **Shape-based (MSE)**: Focuses on the entire curve morphology and transition characteristics

Together, these approaches provide complementary insights into which simulations best capture experimental device behavior.

In [7]:
for d in scaps_data:
    v_sim = d['v(V)']
    j_sim = d['jtot(mA/cm2)']
    
    # Create an interpolator to compare with the exact voltages measured in the lab
    interp_func = interp1d(v_sim, j_sim, kind='linear', fill_value="extrapolate")
    
    # Evaluate only in the overlapping region (common Voltage Range)
    v_min, v_max = max(v_sim.min(), exp_data['v(V)'].min()), min(v_sim.max(), exp_data['v(V)'].max())
    valid_exp = exp_data[(exp_data['v(V)'] >= v_min) & (exp_data['v(V)'] <= v_max)]
    
    if len(valid_exp) > 5:
        j_sim_interp = interp_func(valid_exp['v(V)'])
        mse = mean_squared_error(valid_exp['jtot(mA/cm2)'], j_sim_interp)
        d['mse'] = mse
    else:
        d['mse'] = float('inf')

top_5_mse = sorted(scaps_data, key=lambda x: x.get('mse', float('inf')))[:5]

# Print MSE-based ranking for top 5 matches to terminal
print("\n" + "="*120)
print("TOP 5 MSE MATCHES - SHAPE COMPARISON RESULTS")
print("="*120)
for i, d in enumerate(top_5_mse):
    print(f"\n🔍 Top {i+1} (MSE: {d['mse']:.6f}) - {d['label']}")
    print(f"   Batch Parameters: {d['batch_params']}")
    print(f"   Voc: {d['Voc']:.6f} V | Jsc: {d['Jsc']:.4f} mA/cm² | FF: {d['FF']:.4f} % | η: {d['eta']:.4f} %")
print("="*120)

# ------------------ PLOT 4: TOP 5 VS EXPERIMENTAL (MSE SHAPE) ------------------
fig4 = go.Figure()
fig4.add_trace(go.Scatter(
    x=exp_data['v(V)'], y=exp_data['jtot(mA/cm2)'], 
    mode='lines', name='Experimental', line=dict(color='black', dash='dash', width=3)
))

for i, d in enumerate(top_5_mse):
    v_filtered = d['v(V)']
    j_filtered = d['jtot(mA/cm2)']
    
    hover_text = f"<b>{d['label']}</b><br>MSE: {d['mse']:.4f}"
    fig4.add_trace(go.Scatter(
        x=v_filtered, y=j_filtered, mode='lines', 
        name=f"Top {i+1} (MSE: {d['mse']:.4f})", hoverinfo='text', hovertext=hover_text
    ))

fig4.update_xaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
fig4.update_yaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
fig4.update_layout(
    title='Top 5 Simulations with the Most Similar Curve Shape (MSE)',
    xaxis_title='Voltage (V)', yaxis_title='Current Density (mA/cm2)',
    hovermode='closest', template='plotly_white', width=900, height=600,
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridwidth=1, gridcolor='lightgray'),
    yaxis=dict(showgrid=True, gridwidth=1, gridcolor='lightgray')
)
fig4.show()
fig4.write_image(os.path.join(plots_dir, 'Top5_MSE_Shape.png'), width=900, height=600)
print(f"✅ Saved Plot 4 to {plots_dir}/Top5_MSE_Shape.png")


TOP 5 MSE MATCHES - SHAPE COMPARISON RESULTS

🔍 Top 1 (MSE: 8.135098) - Defect Density = 1.000e+19
   Batch Parameters: [('CdS (L2)>>defect 1>>total defect density [1/cm]', '1.000e+19'), ('Sb2Se3 / CdS (I1)>>Defect 1>>total density [1/cm]', '5.000e+11')]
   Voc: 0.399379 V | Jsc: 21.9596 mA/cm² | FF: 48.3647 % | η: 4.2417 %

🔍 Top 2 (MSE: 10.970033) - Defect Density = 1.000e+18
   Batch Parameters: [('CdS (L2)>>defect 1>>total defect density [1/cm]', '1.000e+18'), ('Sb2Se3 / CdS (I1)>>Defect 1>>total density [1/cm]', '5.000e+11')]
   Voc: 0.401369 V | Jsc: 23.3027 mA/cm² | FF: 49.0434 % | η: 4.5870 %

🔍 Top 3 (MSE: 12.687013) - Defect Density = 1.000e+17
   Batch Parameters: [('CdS (L2)>>defect 1>>total defect density [1/cm]', '1.000e+17'), ('Sb2Se3 / CdS (I1)>>Defect 1>>total density [1/cm]', '5.000e+11')]
   Voc: 0.404051 V | Jsc: 23.5981 mA/cm² | FF: 50.5697 % | η: 4.8217 %

🔍 Top 4 (MSE: 12.965449) - Defect Density = 1.000e+16
   Batch Parameters: [('CdS (L2)>>defect 1>>total defe

✅ Saved Plot 4 to Plots/Top5_MSE_Shape.png


## Section 6: Summary Comparison - Best Matches

This section presents a side-by-side comparison of the experimental J-V curve with the best matching simulations from both approaches:
- **Best Shape Match (MSE)**: Simulation with the most similar curve morphology to experimental data
- **Best Parameter Match**: Simulation with performance parameters (Voc, Jsc, FF, η) closest to targets

In [8]:
# Extract best matches
best_mse = top_5_mse[0]  # Best shape match
best_params = top_5_params[0]  # Best parameter match

# Create comparison figure
fig_summary = go.Figure()

# Add experimental data
fig_summary.add_trace(go.Scatter(
    x=exp_data['v(V)'], y=exp_data['jtot(mA/cm2)'], 
    mode='lines', name='Experimental', 
    line=dict(color='black', dash='dash', width=3),
    hovertemplate='<b>Experimental</b><br>V: %{x:.3f} V<br>J: %{y:.2f} mA/cm²<extra></extra>'
))

# Add best shape match
v_mse = best_mse['v(V)']
j_mse = best_mse['jtot(mA/cm2)']
fig_summary.add_trace(go.Scatter(
    x=v_mse, y=j_mse, 
    mode='lines', name=f'Best Shape Match (MSE: {best_mse["mse"]:.4f})',
    line=dict(color='blue', width=2),
    hovertemplate='<b>Best Shape Match</b><br>V: %{x:.3f} V<br>J: %{y:.2f} mA/cm²<extra></extra>'
))

# Add best parameter match
v_params = best_params['v(V)']
j_params = best_params['jtot(mA/cm2)']
fig_summary.add_trace(go.Scatter(
    x=v_params, y=j_params, 
    mode='lines', name=f'Best Parameter Match (PM: {best_params["param_score"]:.6f})',
    line=dict(color='red', width=2),
    hovertemplate='<b>Best Parameter Match</b><br>V: %{x:.3f} V<br>J: %{y:.2f} mA/cm²<extra></extra>'
))

fig_summary.update_xaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
fig_summary.update_yaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
fig_summary.update_layout(
    title='Experimental vs Best Parameters & Best MSE',
    xaxis_title='Voltage (V)',
    yaxis_title='Current Density (mA/cm²)',
    hovermode='closest',
    template='plotly_white',
    width=1000,
    height=600,
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridwidth=1, gridcolor='lightgray'),
    yaxis=dict(showgrid=True, gridwidth=1, gridcolor='lightgray')
)
fig_summary.show()
fig_summary.write_image(os.path.join(plots_dir, 'Experimental_vs_Best_Parameters_and_Best_MSE.png'), width=1000, height=600)
print(f"✅ Saved Summary Plot to {plots_dir}/Experimental_vs_Best_Parameters_and_Best_MSE.png")

# Create comparison table as visual image
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# Format batch parameters for display
def convert_to_power_format(sci_notation):
    """Convert scientific notation (e.g., 1.00e+10) to power format (e.g., 1*10^10)"""
    import re
    match = re.match(r'([+-]?\d*\.?\d+)e([+-]?\d+)', str(sci_notation), re.IGNORECASE)
    if match:
        mantissa = float(match.group(1))
        exponent = int(match.group(2))
        # Format as "mantissa*10^exponent", removing decimal if it's a whole number
        if mantissa == int(mantissa):
            return f"{int(mantissa)}*10^{exponent}"
        else:
            return f"{mantissa:.0f}*10^{exponent}"
    return str(sci_notation)

def format_batch_params(batch_params):
    """Format batch parameters as a string"""
    params_str = ""
    for param_name, param_val in batch_params:
        # Extract only the part before ">>"
        short_name = param_name.split(">>")[0] if ">>" in param_name else param_name
        # Convert value to power format
        formatted_val = convert_to_power_format(param_val)
        params_str += f"{short_name}: {formatted_val}\n"
    return params_str.rstrip()

# Print batch parameters
mse_batch_str = format_batch_params(best_mse['batch_params'])
params_batch_str = format_batch_params(best_params['batch_params'])

# Display batch parameters line by line
mse_lines = mse_batch_str.split('\n')
params_lines = params_batch_str.split('\n')
max_lines = max(len(mse_lines), len(params_lines))

# Create table data
table_data = [
    ['Metric', 'Experimental', 'Best Shape (MSE)', 'Best Parameters'],
]

# Add batch parameters rows
for i in range(max_lines):
    if i == 0:
        mse_param = mse_lines[i] if i < len(mse_lines) else ""
        params_param = params_lines[i] if i < len(params_lines) else ""
        table_data.append(['Batch Parameters', '', mse_param, params_param])
    else:
        mse_param = mse_lines[i] if i < len(mse_lines) else ""
        params_param = params_lines[i] if i < len(params_lines) else ""
        table_data.append(['', '', mse_param, params_param])

# Add metrics
table_data.append(['Voc (V)', f'{TARGET_VOC:.4f}', f'{best_mse["Voc"]:.4f}', f'{best_params["Voc"]:.4f}'])
table_data.append(['Jsc (mA/cm²)', f'{TARGET_JSC:.4f}', f'{best_mse["Jsc"]:.4f}', f'{best_params["Jsc"]:.4f}'])
table_data.append(['FF (%)', f'{TARGET_FF:.2f}', f'{best_mse["FF"]:.2f}', f'{best_params["FF"]:.2f}'])
table_data.append(['η (%)', f'{TARGET_ETA:.4f}', f'{best_mse["eta"]:.4f}', f'{best_params["eta"]:.4f}'])
table_data.append(['MSE', '', f'{best_mse["mse"]:.6f}', f'{best_params.get("mse", "---")}'])

# Format param score safely
mse_param_score = f"{best_mse.get('param_score', '---'):.6f}" if isinstance(best_mse.get('param_score'), (int, float)) else str(best_mse.get('param_score', '---'))
params_param_score = f"{best_params['param_score']:.6f}"
table_data.append(['Param Score', '', mse_param_score, params_param_score])

# Create figure for table
fig_table, ax = plt.subplots(figsize=(14, 8))
ax.axis('tight')
ax.axis('off')

# Create table
table = ax.table(cellText=table_data, cellLoc='left', loc='center',
                colWidths=[0.15, 0.25, 0.30, 0.30])

# Style the table
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.5)

# Style header row
for i in range(4):
    cell = table[(0, i)]
    cell.set_facecolor('#4472C4')
    cell.set_text_props(weight='bold', color='white')

# Style alternating rows
for i in range(1, len(table_data)):
    for j in range(4):
        cell = table[(i, j)]
        if i % 2 == 0:
            cell.set_facecolor('#E7E6E6')
        else:
            cell.set_facecolor('#F2F2F2')

# Add title
fig_table.suptitle('BEST MATCHES SUMMARY', fontsize=14, fontweight='bold', y=0.98)

# Save table
table_path = os.path.join(plots_dir, 'Summary_Table.png')
plt.savefig(table_path, dpi=300, bbox_inches='tight')
plt.close()
plt.show()
print(f"✅ Saved Summary Table to {plots_dir}/Summary_Table.png")

✅ Saved Summary Plot to Plots/Experimental_vs_Best_Parameters_and_Best_MSE.png
✅ Saved Summary Table to Plots/Summary_Table.png
